4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove
exact duplicates, and sort by order_date.

In [0]:
%python
df = spark.read.table("de_dev.bronze.sales")
df.printSchema()

In [0]:
%python
from pyspark.sql.functions import *

df = spark.read.table("de_dev.bronze.sales")
df = (df
      .dropna(how="all")
      .dropna(subset=["sale_id"])
      .distinct()
      .dropDuplicates(["sale_id"])
      .withColumnRenamed("sale_date", "order_date")
      .withColumn("order_date", coalesce(col("order_date"), current_date()) )
      .fillna({
           "quantity": 1,
           "discount": 0,
           "product_id" :-1,
           "sale_amount" :0,
           "order_status":"pending",
           "payment_method":"Cash"
        })
      .drop("source_file_name","file_path","file_name","ingestion_timestamp")
      .orderBy("order_date")
      )

# df.display()




5. Perform an aggregation (revenue by category or region) and a join against a second small reference
table (e.g., customers or regions).

-> I first calculated the total revenue for each customer using groupBy() and sum(). Then I joined this result with the customer_df using customer_id. This adds customer information, such as the customer's name or email, to the revenue result.

In [0]:
%python
from pyspark.sql.functions import *

sales_df = spark.read.table("de_dev.bronze.sales")
customer_df = spark.read.table("de_dev.bronze.customers")

revenue_df = (sales_df
              .groupBy("customer_id")
              .agg(sum("sale_amount").alias("total_revenue")) 
            )

# joining with the customers 
final_df = (revenue_df
            .join(customer_df, "customer_id")
            )

# final_df.display()


6. Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request
describing what changed and why.

-> To do the all such creating branch to mergin it to main i'll follow the following steps

1.In Databricks, I opened my cloned Git repository and created a new feature branch, for example feature/update-cleaning.

2.I changed the cleaning logic in the notebook, such as handling null values differently or adding a new validation rule.

3.I committed the changes to the feature branch with a clear commit message.

4.I pushed the feature branch to the remote Git repository.

5.I opened a Pull Request (PR) from the feature branch to the main branch.

6.In the PR description, I explained what I changed and why, so the changes could be reviewed before merging.